CISC 340 Lab 4

Class label is Play and four feature columns are Outlook, Temprature, Humidity, Windy. Total rows is 99. 

In [1]:
!pip install pandas openpyxl

In [4]:
df = pd.read_excel("C:\\Users\\bhavi\\Downloads\\24CISC340_6_Data.xlsx")

In [5]:
df.head()

,Outlook,Temperature,Humidity,Windy,Play
0,Overcast,Mild,High,No,No
1,Sunny,Mild,High,No,Yes
2,Sunny,Hot,High,No,Yes
3,Sunny,Hot,Normal,No,Yes
4,Overcast,Cold,Normal,Yes,Yes


In [6]:
print(df['Outlook'].value_counts())

Outlook
Rainy       36
Sunny       35
Overcast    28
Name: count, dtype: int64


In [7]:
print(df['Temperature'].value_counts())

Temperature
Hot     38
Cold    36
Mild    25
Name: count, dtype: int64


In [8]:
print(df['Humidity'].value_counts())

Humidity
High      50
Normal    49
Name: count, dtype: int64


In [9]:
print(df['Windy'].value_counts())

Windy
No     59
Yes    40
Name: count, dtype: int64


In [10]:
print(df['Play'].value_counts())

Play
No     50
Yes    49
Name: count, dtype: int64


In [11]:
pd.crosstab(df['Outlook'], df['Play'])

Play,No,Yes
Outlook,,
Overcast,15,13
Rainy,18,18
Sunny,17,18


In [12]:
pd.crosstab(df['Outlook'], df['Play'], normalize='columns')

Play,No,Yes
Outlook,,
Overcast,0.30,0.265306
Rainy,0.36,0.367347
Sunny,0.34,0.367347


In [13]:
pd.crosstab(df['Temperature'], df['Play'])

Play,No,Yes
Temperature,,
Cold,18,18
Hot,20,18
Mild,12,13


In [14]:
pd.crosstab(df['Temperature'], df['Play'], normalize='columns')

Play,No,Yes
Temperature,,
Cold,0.36,0.367347
Hot,0.40,0.367347
Mild,0.24,0.265306


In [15]:
pd.crosstab(df['Humidity'], df['Play'])

Play,No,Yes
Humidity,,
High,25,25
Normal,25,24


In [16]:
pd.crosstab(df['Humidity'], df['Play'], normalize='columns')

Play,No,Yes
Humidity,,
High,0.5,0.510204
Normal,0.5,0.489796


In [17]:
pd.crosstab(df['Windy'], df['Play'])

Play,No,Yes
Windy,,
No,32,27
Yes,18,22


In [18]:
pd.crosstab(df['Windy'], df['Play'], normalize='columns')

Play,No,Yes
Windy,,
No,0.64,0.55102
Yes,0.36,0.44898


In [19]:
import pandas as pd
# Prior probabilities
P_yes = 0.495
P_no = 0.505
# Conditional probabilities
# Format: P(feature_value | class)
prob_outlook = {
    'Sunny': {'Yes': 0.3673, 'No': 0.34},
    'Overcast': {'Yes': 0.2653, 'No': 0.30},
    'Rainy': {'Yes': 0.3673, 'No': 0.36}
}
prob_temperature = {
    'Mild': {'Yes': 0.367, 'No': 0.24},
    'Hot': {'Yes': 0.367, 'No': 0.40},
    'Cold': {'Yes': 0.2653, 'No': 0.36}
}
prob_humidity = {
    'High': {'Yes': 0.5102, 'No': 0.5},
    'Normal': {'Yes': 0.4898, 'No': 0.5}
}
prob_windy = {
    False: {'Yes': 0.551, 'No': 0.64},
    True: {'Yes': 0.449, 'No': 0.36}
}
# New records to classify
records = [
    {'Outlook': 'Sunny', 'Temperature': 'Mild', 'Humidity': 'Normal', 'Windy': False},
    {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Windy': True},
    {'Outlook': 'Rainy', 'Temperature': 'Hot', 'Humidity': 'High', 'Windy': True}
]
# Naive Bayes prediction
predictions = []
for rec in records:
    # Probability for Yes
    P_rec_yes = (
        P_yes *
        prob_outlook[rec['Outlook']]['Yes'] *
        prob_temperature[rec['Temperature']]['Yes'] *
        prob_humidity[rec['Humidity']]['Yes'] *
        prob_windy[rec['Windy']]['Yes']
    )
    # Probability for No
    P_rec_no = (
        P_no *
        prob_outlook[rec['Outlook']]['No'] *
        prob_temperature[rec['Temperature']]['No'] *
        prob_humidity[rec['Humidity']]['No'] *
        prob_windy[rec['Windy']]['No']
    )
    
    # Predict class
    predicted_class = 'Yes' if P_rec_yes > P_rec_no else 'No'
    
    # Store results
    predictions.append({
        **rec,
        'P_Yes': round(P_rec_yes, 5),
        'P_No': round(P_rec_no, 5),
        'Predicted_Play': predicted_class
    })

# Convert to DataFrame for neat display
df_predictions = pd.DataFrame(predictions)
print(df_predictions)

    Outlook Temperature Humidity  Windy    P_Yes     P_No Predicted_Play
0     Sunny        Mild   Normal  False  0.01801  0.01319            Yes
1  Overcast         Hot   Normal   True  0.01060  0.01091             No
2     Rainy         Hot     High   True  0.01529  0.01309            Yes


In [33]:
import pandas as pd
import math

class ID3:
    def __init__(self):
        self.tree = None

    # entropy
    def _entropy(self, data):
        target = data['Play']
        counts = target.value_counts()
        total = len(target)

        entropy = 0
        for count in counts:
            p = count / total
            entropy -= p * math.log2(p)
        return entropy

    # informationgain
    def _info_gain(self, data, feature):
        total_entropy = self._entropy(data)
        values = data[feature].unique()

        weighted_entropy = 0
        for v in values:
            subset = data[data[feature] == v]
            weight = len(subset) / len(data)
            weighted_entropy += weight * self._entropy(subset)

        return total_entropy - weighted_entropy

    # buildtree
    def _build_tree(self, data, features):
        target = data['Play']

        if len(target.unique()) == 1:
            return target.iloc[0]

        if len(features) == 0:
            return target.mode()[0]

        gains = [self._info_gain(data, f) for f in features]
        best_feature = features[gains.index(max(gains))]

        tree = {best_feature: {}}

        for value in data[best_feature].unique():
            subset = data[data[best_feature] == value]

            if subset.empty:
                tree[best_feature][value] = target.mode()[0]
            else:
                remaining = [f for f in features if f != best_feature]
                tree[best_feature][value] = self._build_tree(subset, remaining)

        return tree

    # train
    def train(self, rawData):
        features = [col for col in rawData.columns if col != 'Play']
        self.tree = self._build_tree(rawData, features)

    # classify
    def classify(self, record):
        tree = self.tree

        while isinstance(tree, dict):
            feature = list(tree.keys())[0]
            value = str(record.get(feature)).lower()

            keys = {str(k).lower(): k for k in tree[feature].keys()}

            if value in keys:
                tree = tree[feature][keys[value]]
            else:
                return "Unknown"

        return tree

    # display tree
    def displayTree(self, tree=None, indent=""):
        if tree is None:
            tree = self.tree

        if not isinstance(tree, dict):
            print(indent + "-> " + str(tree))
            return

        for feature, branches in tree.items():
            for value, subtree in branches.items():
                print(indent + f"{feature} = {value}")
                self.displayTree(subtree, indent + "   ")

In [34]:
# Load dataset
df = pd.read_excel("C:\\Users\\bhavi\\Downloads\\24CISC340_6_Data.xlsx")

# CLEAN DATA
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]  # remove empty columns
df = df.dropna()  # remove missing rows

# Ensure correct column order
df = df[['Play', 'Outlook', 'Temperature', 'Humidity', 'Windy']]

# Fix Windy values
df['Windy'] = df['Windy'].astype(str).str.lower()

# Train model
model = ID3()
model.train(df)

# Display decision tree
print("Decision Tree:")
model.displayTree()

# Test records
records = [
    {"Outlook": "Sunny", "Temperature": "Mild", "Humidity": "Normal", "Windy": "no"},
    {"Outlook": "Overcast", "Temperature": "Hot", "Humidity": "Normal", "Windy": "yes"},
    {"Outlook": "Rainy", "Temperature": "Hot", "Humidity": "High", "Windy": "yes"}
]

# Classify
print("\nPredictions:")
for r in records:
    print(r, "->", model.classify(r))

Decision Tree:
Windy = no
   Outlook = Overcast
      Temperature = Mild
         -> No
      Temperature = Cold
         Humidity = High
            -> Yes
         Humidity = Normal
            -> No
      Temperature = Hot
         Humidity = High
            -> No
         Humidity = Normal
            -> Yes
   Outlook = Sunny
      Temperature = Mild
         Humidity = High
            -> No
         Humidity = Normal
            -> Yes
      Temperature = Hot
         Humidity = High
            -> No
         Humidity = Normal
            -> Yes
      Temperature = Cold
         Humidity = High
            -> Yes
         Humidity = Normal
            -> No
   Outlook = Rainy
      Temperature = Cold
         Humidity = High
            -> No
         Humidity = Normal
            -> No
      Temperature = Hot
         Humidity = High
            -> No
         Humidity = Normal
            -> No
      Temperature = Mild
         -> Yes
Windy = yes
   Temperature = Cold
      

The Naive Bayes classifier and the ID3 Decision Tree do not necessarily produce the same results because they are based on different learning principles. Naive Bayes is a probabilistic model that assumes independence among features and calculates class probabilities using all attributes simultaneously. In contrast, ID3 builds a decision tree using information gain and creates rule-based splits. It is capturing relationships between features. Due to these differences, especially the independence assumption in Naive Bayes and the hierarchical structure of ID3, their predictions can vary. They may produce similar results in simple cases, but they will not always be the same in general.